In [3]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.tokenize import sent_tokenize
from nltk.corpus import stopwords
from rouge_score import rouge_scorer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import networkx as nx
import joblib

In [4]:
import string
def clean_text(text):

    if not isinstance(text, str):
        return ""

    text = re.sub(r"<.*?>", " ", text)

    text = re.sub(r"http\S+|www\S+|https\S+", " ", text)

    text = re.sub(
        "[" 
        u"\U0001F600-\U0001F64F"
        u"\U0001F300-\U0001F5FF"
        u"\U0001F680-\U0001F6FF"
        u"\U0001F1E0-\U0001F1FF"
        "]+", 
        "", 
        text
    )

    text = re.sub(r"[^a-zA-Z0-9\s.,!?]", " ", text)

    allowed = set(string.ascii_letters + string.digits + " .,!?")
    text = "".join(ch for ch in text if ch in allowed)

    text = text.lower()

    text = " ".join(text.split())

    return text

In [5]:
df = pd.read_csv("D:/Final Project/Data/news.tsv", sep="\t")

df["text"] = df["Headline"].fillna("") + " " + df["News body"].fillna("")
df = df.rename(columns={"Category": "label"}).dropna()

df["clean_text"] = df["text"].apply(clean_text)

df["full_text"] = df["text"]

In [6]:
import nltk
nltk.download("punkt")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ADMIN\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ADMIN\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

Sentence & word tokenization

In [7]:
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

def tokenize_and_remove_stopwords(text):
    words = word_tokenize(text)
    words = [w for w in words if w not in stop_words and w.isalpha()]
    return words

In [8]:
df["tokens"] = df["clean_text"].apply(tokenize_and_remove_stopwords)

In [9]:
MAX_ARTICLE_TOKENS = 512
MAX_SUMMARY_TOKENS = 128

def truncate_text(text, max_tokens):
    tokens = word_tokenize(text)
    tokens = tokens[:max_tokens]
    return " ".join(tokens)

df["article_trunc"] = df["clean_text"].apply(
    lambda x: truncate_text(x, MAX_ARTICLE_TOKENS)
)

In [10]:
df["article_trunc"]

0         predicting atlanta united s lineup against col...
1         mitch mcconnell dc statehood push is full bore...
2         home in north highlands damaged by fire north ...
3         meghan mccain blames liberal media and third w...
4         today in history aug 1 1714 george i becomes k...
                                ...                        
113757    hope who ? alyssa naeher s penalty save sends ...
113758    chris sale explains what specifically has gone...
113759    raptor fans jam streets to celebrate 1st nba t...
113760    judge won t allow flynn to fire his attorneys ...
113761    worley thinks he and conley will rival greates...
Name: article_trunc, Length: 113704, dtype: object

Vectorization / Feature Representations

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words="english"
)

tfidf_matrix = tfidf_vectorizer.fit_transform(df["clean_text"])

Extractive Summarization (TF-IDF Baseline)

In [12]:
from nltk.tokenize import sent_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

def tfidf_extractive_summary(text, top_k=3):
    sentences = sent_tokenize(text)

    if len(sentences) <= top_k:
        return text

    vectorizer = TfidfVectorizer(stop_words="english")
    tfidf_matrix = vectorizer.fit_transform(sentences)

    scores = tfidf_matrix.sum(axis=1).A1
    top_sentence_ids = scores.argsort()[-top_k:]
    top_sentence_ids.sort()

    summary = " ".join([sentences[i] for i in top_sentence_ids])
    return summary

In [13]:
df["tfidf_summary"] = df["full_text"].apply(
    lambda x: tfidf_extractive_summary(x, top_k=3)
)

In [14]:
df["tfidf_summary"]

0         We've seen how he rotates (or doesn't rotate) ...
1         Mitch McConnell: DC statehood push is 'full bo...
2         Home In North Highlands Damaged By Fire NORTH ...
3         Meghan McCain blames 'liberal media' and 'thir...
4         1798: Battle of Nile begins Battle of Nile, al...
                                ...                        
113757    No, when the final whistle sounded, the entire...
113758    In his last start before the All-Star break, S...
113759    Raptor fans jam streets to celebrate 1st NBA t...
113760    Attorneys for President Trump's former nationa...
113761    The kind of season they had overall lends litt...
Name: tfidf_summary, Length: 113704, dtype: object

In [15]:
df["tfidf_summary"][0]

"We've seen how he rotates (or doesn't rotate) in CONCACAF Champions League play, but that's a bit different because CCL was clearly a priority for the club. We got one glimpse of U.S. Open Cup rotation last week when the team played a home game as the visiting team against the Charleston Battery, but will things change on the actual road against an MLS club? Miles Robinson is still young and probably isn't in need of much rest, and as far as international players go, you probably want those to be your most indispensable and/or attacking options in case things go awry."

TEXT RANK SUMMARIZER

In [16]:
def textrank_summarize(text, top_n=3):
    cleaned = clean_text(text)
    sentences = sent_tokenize(cleaned)

    if len(sentences) <= top_n:
        return " ".join(sentences)

    vectorizer = TfidfVectorizer()
    vectors = vectorizer.fit_transform(sentences).toarray()

    sim_matrix = cosine_similarity(vectors)

    nx_graph = nx.from_numpy_array(sim_matrix)
    scores = nx.pagerank(nx_graph)

    ranked = sorted(((scores[i], s) for i, s in enumerate(sentences)), reverse=True)
    summary = " ".join([s for _, s in ranked[:top_n]])

    return summary

In [17]:
def tfidf_summarize(text, top_n=3):
    cleaned = clean_text(text)
    sentences = sent_tokenize(cleaned)

    if len(sentences) <= top_n:
        return " ".join(sentences)

    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(sentences)

    scores = tfidf_matrix.mean(axis=1).A.flatten()

    ranked_idx = np.argsort(scores)[::-1]
    selected = [sentences[i] for i in ranked_idx[:top_n]]

    return " ".join(selected)

In [18]:
def reference_summary(text):
    sents = sent_tokenize(clean_text(text))
    return " ".join(sents[:2])

Evaluation

In [19]:
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

def evaluate_model(summarizer_fn, df, samples=50):

    rouge1_scores, rouge2_scores, rougeL_scores = [], [], []

    for i in range(min(samples, len(df))):

        text = df.iloc[i]["full_text"]
        ref = reference_summary(text)
        pred = summarizer_fn(text)

        scores = scorer.score(ref, pred)

        rouge1_scores.append(scores["rouge1"].fmeasure)
        rouge2_scores.append(scores["rouge2"].fmeasure)
        rougeL_scores.append(scores["rougeL"].fmeasure)

    return {
        "rouge1": np.mean(rouge1_scores),
        "rouge2": np.mean(rouge2_scores),
        "rougeL": np.mean(rougeL_scores),
    }

In [20]:
print("Evaluating TextRank...")
textrank_scores = evaluate_model(textrank_summarize, df)

print("Evaluating TF-IDF...")
tfidf_scores = evaluate_model(tfidf_summarize, df)

Evaluating TextRank...
Evaluating TF-IDF...


In [21]:
OUTPUT_CSV = "summarization_comparison_results.csv"

new_results = pd.DataFrame([
    {
        "Model": "TextRank",
        "rouge1": textrank_scores["rouge1"],
        "rouge2": textrank_scores["rouge2"],
        "rougeL": textrank_scores["rougeL"],
        "rougeLsum": textrank_scores["rougeL"],
        "Average Score": np.mean([
            textrank_scores["rouge1"],
            textrank_scores["rouge2"],
            textrank_scores["rougeL"],
        ])
    },
    {
        "Model": "TF-IDF",
        "rouge1": tfidf_scores["rouge1"],
        "rouge2": tfidf_scores["rouge2"],
        "rougeL": tfidf_scores["rougeL"],
        "rougeLsum": tfidf_scores["rougeL"],
        "Average Score": np.mean([
            tfidf_scores["rouge1"],
            tfidf_scores["rouge2"],
            tfidf_scores["rougeL"],
        ])
    }
])

new_results = new_results.round(4)

try:
    old_df = pd.read_csv(OUTPUT_CSV)
    final_df = pd.concat([old_df, new_results], ignore_index=True)
except FileNotFoundError:
    final_df = new_results

final_df.to_csv(OUTPUT_CSV, index=False)

print("\nEvaluation Completed.\nSaved to:", OUTPUT_CSV)


Evaluation Completed.
Saved to: summarization_comparison_results.csv


In [22]:
import pandas as pd
df1=pd.read_csv("D:/Final Project/Summarization/summarization_comparison_results.csv")
df1

,Model,rouge1,rouge2,rougeL,rougeLsum,Average Score
0,TextRank,0.5028,0.3924,0.4402,0.4402,0.4451
1,TF-IDF,0.4836,0.3779,0.4144,0.4144,0.4253
